# Music Genre Classification — Data Transformation & Preprocessing

## Objective

This notebook covers the **Data Transformation** phase of our music genre 
classification project, bridging our completed EDA (Sprint 1) and the 
upcoming ML modeling phase.

Our pipeline follows three stages:
1. **EDA** — completed (class distribution, waveforms, spectrograms, MFCCs, 
   corrupted/duplicate file detection)
2. **Data Transformation** — this notebook (cleaning validation + preprocessing pipeline)
3. **ML Model** — PyTorch-based classifier (next phase)

## Goal

By the end of this notebook, we will have a working PyTorch `Dataset` and 
`DataLoader` that:
- Reads clean, verified track metadata (file path, genre label, split 
  assignment) from our PostgreSQL database (`vw_clean_tracks`), which already 
  excludes corrupted and duplicate files identified during EDA
- Loads each audio file at its native, verified sample rate (22050 Hz — 
  confirmed consistent across all 999 readable files during EDA)
- Trims/pads each clip to a standard 30-second length, so every MFCC array 
  has a consistent shape
- Extracts MFCCs as full 2D arrays (20 coefficients × time), **not averaged**, 
  since our model is a PyTorch CNN that needs the temporal structure of the 
  audio, not a single summary vector
- Normalizes MFCC values (fit only on the training split, to avoid data leakage)
- Applies data augmentation (time shifting, noise, frequency masking) 
  **only to the training split**
- Returns a batch of ready-to-train tensors with correct shape and value 
  range when passed through a `DataLoader`

## Dataset

We use only the `genres_original/` audio files from this package. The 
pre-extracted feature CSVs and spectrogram images included in the download 
are **not** used as model input they were referenced only during EDA for 
exploratory analysis, since our model computes its own MFCC features 
directly from the raw audio.

Reproducibility: all random operations in this notebook use `seed = 42` 
(numpy, torch, and Python's `random` module).

In [4]:
# --- Standard library ---
import os
import random
from pathlib import Path

# --- Data & numeric ---
import numpy as np
import pandas as pd

# --- Audio processing ---
import librosa
import soundfile as sf

# --- Database connection ---
import psycopg2
from dotenv import load_dotenv

# --- PyTorch ---
import torch
from torch.utils.data import Dataset, DataLoader

# --- SQLAlchemy ---
from sqlalchemy import create_engine

In [ ]:
## Set random seeds for reproducibility 
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Conexión SQL (971 clean_tracks)

In [ ]:

# load environment variables from .env file 
load_dotenv()

# read DB connection details from environment variables
DB_NAME = os.getenv("DB_NAME", "music_genre_db")  # database name, default to music_genre_db
DB_USER = os.getenv("DB_USER", "ingxrodriguez")    # postgres user
DB_PASSWORD = os.getenv("DB_PASSWORD")              # postgres password, must come from .env, never hardcoded
DB_HOST = os.getenv("DB_HOST", "localhost")         # db host
DB_PORT = os.getenv("DB_PORT", "5432")              # db port

# create a SQLAlchemy engine using those credentials
engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# query the clean tracks view: excludes corrupted_flagged and duplicate_flagged rows
query = "SELECT track_id, file_path, label, genre_id, split FROM vw_clean_tracks;"

# read the query result directly into a pandas DataFrame, using the engine 
clean_tracks_df = pd.read_sql(query, engine)

# quick sanity check: confirm row count and preview the first few rows
print("Total clean tracks:", len(clean_tracks_df))
clean_tracks_df.head()

Total clean tracks: 971


,track_id,file_path,label,genre_id,split
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val


# Trim/pad a 30 seg

### Why we trim/pad to 30 seconds

Our EDA showed that GTZAN clips are almost all ~30 seconds long, but not 
perfectly uniform — a few are slightly shorter or longer. Since our model 
extracts MFCCs as full 2D arrays (coefficients × time), every clip must 
have the exact same number of audio samples; otherwise the resulting MFCC 
arrays will have different time dimensions and can't be batched together 
by the DataLoader.

To fix this, we standardize every clip to exactly 30 seconds (22050 Hz × 30 
= 661,500 samples):
- Clips longer than 30s are **trimmed** down to the target length.
- Clips shorter than 30s are **padded** with silence (zeros) at the end.

Original files are never modified — trimmed/padded copies are saved to a 
separate `processed/` folder, preserving the raw dataset as our permanent 
reference.

In [7]:
# native sample rate confirmed identical across all readable files during EDA (22050 Hz)
TARGET_SR = 22050

# standard clip length decided during cleaning: 30 seconds (matches original GTZAN clip length)
TARGET_DURATION_SEC = 30

# convert target duration to number of samples (30 sec * 22050 samples/sec)
TARGET_LENGTH_SAMPLES = TARGET_SR * TARGET_DURATION_SEC

# root folder where original, untouched audio files live
DATA_ROOT = Path("../Data_Music")

# new folder for trimmed/padded copies -- originals are never overwritten
PROCESSED_DIR = DATA_ROOT / "processed"


def trim_or_pad(y, target_length):
    """Trim audio array to target_length, or pad with zeros (silence) if shorter."""
    current_length = len(y)  # number of samples in the loaded audio

    if current_length > target_length:
        # clip is longer than target: cut it down to exactly target_length samples
        return y[:target_length]
    elif current_length < target_length:
        # clip is shorter than target: pad the end with zeros (silence) up to target_length
        pad_amount = target_length - current_length
        return np.pad(y, (0, pad_amount), mode="constant")
    else:
        # already exactly the target length, no change needed
        return y


# will store the new, trimmed file path for each track so we can save it back to the dataframe
processed_paths = []

# loop through every clean track from our SQL query
for row in clean_tracks_df.itertuples(index=False):
    # load the original audio at its verified native sample rate (sr=None reads actual rate, doesn't impose one)
    y, sr = librosa.load(row.file_path, sr=None)

    # apply trim/pad so every clip has exactly TARGET_LENGTH_SAMPLES samples
    y_fixed = trim_or_pad(y, TARGET_LENGTH_SAMPLES)

    # build the output subfolder path, mirroring genre structure: processed/<label>/
    genre_subdir = PROCESSED_DIR / row.label
    genre_subdir.mkdir(parents=True, exist_ok=True)  # create folder if it doesn't exist yet

    # build the output file path using the same filename as the original
    out_path = genre_subdir / Path(row.file_path).name

    # write the trimmed/padded audio to disk (original file is untouched)
    sf.write(out_path, y_fixed, sr)

    # record the new path so we can update our dataframe afterward
    processed_paths.append(str(out_path))

# add the processed file path as a new column, keeping the original file_path column intact
clean_tracks_df["processed_path"] = processed_paths

# confirm how many files were processed
print(f"Trimmed/padded {len(clean_tracks_df)} files to {TARGET_DURATION_SEC} sec each.")
clean_tracks_df.head()

Trimmed/padded 971 files to 30 sec each.


,track_id,file_path,label,genre_id,split,processed_path
0,16,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00015.wav
1,41,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00040.wav
2,3,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,train,../Data_Music/processed/blues/blues.00002.wav
3,78,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00077.wav
4,28,/Users/ingxrodriguez/music-genre-classificatio...,blues,1,val,../Data_Music/processed/blues/blues.00027.wav


# Load audio, calculate MFCC of 20 coefficients, return tensor + label

In [17]:
class GTZANDataset(Dataset):
    """PyTorch Dataset that loads a trimmed/padded audio clip and returns its normalized MFCC array + genre label."""

    def __init__(self, metadata_df, split, n_mfcc=20, mfcc_mean=None, mfcc_std=None):
        # keep only the rows belonging to this split ("train", "val", or "test")
        self.df = metadata_df[metadata_df["split"] == split].reset_index(drop=True)

        # store the split name, useful later to decide whether to apply augmentation (train only)
        self.split = split

        # number of MFCC coefficients to extract per frame (standardized to 20, matches EDA)
        self.n_mfcc = n_mfcc

        # normalization stats -- MUST be computed from the train split only, then reused for all splits
        self.mfcc_mean = mfcc_mean
        self.mfcc_std = mfcc_std

    def __len__(self):
        # total number of tracks available in this split
        return len(self.df)

    def __getitem__(self, idx):
        # look up the row for this index in our filtered split dataframe
        row = self.df.iloc[idx]

        # load the already trimmed/padded audio file (sr=None preserves the verified native rate)
        y, sr = librosa.load(row["processed_path"], sr=None)

        # compute the full 2D MFCC array: shape = (n_mfcc, T) -- NOT averaged over time
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=self.n_mfcc)

        # convert the MFCC numpy array into a PyTorch tensor (float32, standard for model input)
        mfcc_tensor = torch.tensor(mfcc, dtype=torch.float32)

        # apply normalization using the train-only mean/std, if provided
        if self.mfcc_mean is not None and self.mfcc_std is not None:
            mfcc_tensor = (mfcc_tensor - self.mfcc_mean) / self.mfcc_std

        # genre_id is our numeric label (matches the genre_id column from music_genre table)
        label = torch.tensor(row["genre_id"], dtype=torch.long)

        # return the normalized MFCC tensor and its label -- this is what the DataLoader will batch together
        return mfcc_tensor, label

In [13]:
train_dataset = GTZANDataset(clean_tracks_df, split="train")
print("Train size:", len(train_dataset))

Train size: 677


In [14]:
mfcc_sample, label_sample = train_dataset[0]
print("MFCC shape:", mfcc_sample.shape)
print("Label:", label_sample)

MFCC shape: torch.Size([20, 1292])
Label: tensor(1)


In [15]:
mfcc_sample, label_sample = train_dataset[0]
print("MFCC shape:", mfcc_sample.shape)
print("Label:", label_sample)

MFCC shape: torch.Size([20, 1292])
Label: tensor(1)


# Summary: GTZANDataset Class

This class wraps our entire feature-extraction step into a single reusable 
PyTorch `Dataset`. Given a split ("train", "val", or "test"), it:

- Filters `clean_tracks_df` down to only that split's tracks
- On each call to `__getitem__`, loads the trimmed/padded 30-second audio 
  file and computes its MFCC as a **full 2D array** (20 coefficients × time), 
  not an averaged summary vector — preserving the temporal structure our 
  CNN needs
- Returns the MFCC as a tensor along with its numeric genre label 
  (`genre_id`)

Verified output: `MFCC shape: torch.Size([20, 1292])` — 20 MFCC 
coefficients, 1292 time frames, consistent across every track thanks to 
the 30-second standardization from the previous step.

# Normalization 

In [18]:
# create a temporary train-only dataset (no normalization applied yet) to compute stats from
train_dataset_raw = GTZANDataset(clean_tracks_df, split="train")

# accumulate every training MFCC array into a list so we can compute overall mean/std
all_train_mfccs = []

# loop through every track in the train split only
for i in range(len(train_dataset_raw)):
    mfcc_tensor, _ = train_dataset_raw[i]  # ignore the label, we only need the MFCC values here
    all_train_mfccs.append(mfcc_tensor)

# stack all MFCC tensors into one big tensor: shape (num_train_tracks, 20, 1292)
stacked_train_mfccs = torch.stack(all_train_mfccs)

# compute a single global mean and std across all train MFCC values (across tracks and time frames)
MFCC_MEAN = stacked_train_mfccs.mean()
MFCC_STD = stacked_train_mfccs.std()

# print the computed stats so we can document/reproduce them later
print("Train MFCC mean:", MFCC_MEAN.item())
print("Train MFCC std:", MFCC_STD.item())

Train MFCC mean: -0.7768932580947876
Train MFCC std: 51.58198547363281
